# 오늘의집 원룸 인테리어 이미지 크롤링

[오늘의집 피드](https://ohou.se/cards/feed)에서 로그인 없이 이미지를 수집합니다.
무드 분포를 보강하기 위해 **검색어 8개(QUERIES)를 순서대로 순회**하며, 검색어당 최대 `MAX_IMAGES`(100)장씩 받습니다.

- **백그라운드 실행** (Chrome 최소화 — headless는 사이트에서 차단됨)
- **위에서부터 한 줄(4개)씩** 다운로드
- 화면에 더 받을 이미지가 없을 때만 스크롤
- 검색어별로 `images/{검색어}_오늘의집/`에 따로 저장 (나중에 `images/final/`로 합치기)
- **`images/` 전체(다른 색상/키워드 폴더 포함)에 이미 있는 사진은 콘텐츠 해시로 걸러서 중복 저장 안 함**
- 검색어 하나가 실패(차단/타임아웃)해도 나머지 검색어는 이어서 진행

In [1]:
# 크롤링 의존성: selenium(브라우저), webdriver-manager(ChromeDriver), requests(이미지 다운로드)
%pip install selenium webdriver-manager requests

  Using cached webdriver_manager-4.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.meta

In [ ]:
import re  # 썸네일 URL → 고해상도 변환
import time  # 스크롤·대기
import hashlib  # URL → 파일명 해시
from pathlib import Path  # 저장 경로
from urllib.parse import urlparse, quote  # URL 확장자 추출, 검색어 URL 인코딩

import requests  # HTTP 이미지 다운로드
from selenium import webdriver  # Chrome 브라우저 제어
from selenium.webdriver.chrome.service import Service  # ChromeDriver 서비스
from selenium.webdriver.chrome.options import Options  # headless/최소화 등 옵션
from selenium.webdriver.common.by import By  # CSS 셀렉터
from selenium.webdriver.common.action_chains import ActionChains  # 실제 마우스 휠처럼 스크롤
from selenium.webdriver.support.ui import WebDriverWait  # 요소 로딩 대기
from selenium.webdriver.support import expected_conditions as EC  # 대기 조건
from webdriver_manager.chrome import ChromeDriverManager  # ChromeDriver 자동 설치

In [ ]:
# === 설정 ===
# 현재 무드 분포(파스텔·우드·빈티지는 이미 충분, 럭셔리 모던·밝은 에어리 등은 부족)를
# 보강하기 위한 검색어 8개. 검색어별로 images/{검색어}_오늘의집/ 에 따로 저장한 뒤
# 나중에 images/final/로 합친다.
QUERIES = [
    "하이엔드 인테리어",
    "화이트톤 인테리어",
    "코지 원룸",
    "그레이톤 인테리어",
    "미니멀 룸",
    "빈티지",
    "파스텔",
    "우드",
]

def build_search_url(query: str) -> str:
    """검색어 → 오늘의집 피드 검색 URL."""
    return f"https://ohou.se/cards/feed?query={quote(query)}"

def build_save_dir(query: str) -> Path:
    """검색어 → 저장 폴더 (기존 '모던_인테리어_오늘의집' 폴더명 규칙과 동일)."""
    return Path(f"images/{query.replace(' ', '_')}_오늘의집")

# 검색결과 페이지는 article.card-search-item 구조. 중간 래퍼 클래스(content_link)에
# 해시가 붙어 안 맞을 수 있어, img 자체의 class="image"로 바로 스코프 (프로필 사진 제외 유지)
CARD_IMG_SELECTOR = "article.card-search-item img.image"
ROW_SIZE = 4           # 한 줄(행)당 카드 수
SCROLL_PAUSE = 3.5     # 스크롤 후 대기(초) — 검색결과 페이지는 다음 배치 로딩이 느려서 늘림
MAX_NO_NEW_SCROLLS = 6 # 연속으로 새 콘텐츠 없다고 판단하기 전 재시도 횟수
PAGE_LOAD_WAIT = 30    # 카드 로딩 대기(초)
MAX_IMAGES = 100       # 검색어 1개당 최대 다운로드 개수

In [ ]:
def create_driver():
    """Chrome 드라이버 생성 (최소화 모드, 봇 탐지 우회)."""
    options = Options()
    # headless는 오늘의집에서 Access Denied → 최소화로 백그라운드 실행
    options.add_argument("--start-minimized")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    # 최소화 상태로 오래 두면 크롬이 백그라운드 탭으로 취급해 스로틀링(성능 억제)해서
    # 응답이 느려지다가 결국 Selenium이 ReadTimeoutError로 끊기는 문제 방지
    options.add_argument("--disable-backgrounding-occluded-windows")
    options.add_argument("--disable-renderer-backgrounding")
    options.add_argument("--disable-background-timer-throttling")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),  # ChromeDriver 자동 설치
        options=options,
    )
    # 위 옵션으로도 완전히 못 막는 스로틀링이 남아있어, 크롬이 명령에 120초(기본값) 안에
    # 응답 못하면 urllib3 ReadTimeoutError로 크롤링 전체가 죽는다 → 타임아웃 자체를 늘려서 흡수
    driver.command_executor._client_config.timeout = 300
    driver.execute_script(  # navigator.webdriver 숨김
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def dismiss_popups(driver):
    """로그인/앱 유도 팝업 등 닫기 시도."""
    for selector in (
        "button[aria-label='Close']",
        "button[aria-label='닫기']",
        "[class*='close']",
    ):
        try:
            for btn in driver.find_elements(By.CSS_SELECTOR, selector):
                if btn.is_displayed():
                    btn.click()
                    time.sleep(0.5)
        except Exception:
            pass


def wait_for_cards(driver, timeout=PAGE_LOAD_WAIT):
    """검색 결과 카드가 로드될 때까지 대기."""
    print(f"페이지 로딩 대기 중... (최대 {timeout}초)")
    dismiss_popups(driver)

    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, CARD_IMG_SELECTOR)
            )
        )
    except Exception:
        title = driver.title
        if "Access Denied" in title:
            raise RuntimeError(
                "오늘의집 접근 차단(Access Denied). "
                "headless가 아닌 최소화 모드인지, 인터넷 연결을 확인하세요."
            ) from None

        # 디버깅용: 실제로 자동화 브라우저가 뭘 받았는지 스크린샷 + 태그 개수로 확인
        debug_dir = Path("debug")
        debug_dir.mkdir(exist_ok=True)
        screenshot_path = debug_dir / "crawl_timeout.png"
        try:
            driver.save_screenshot(str(screenshot_path))
        except Exception:
            screenshot_path = None

        counts = {
            "article": len(driver.find_elements(By.TAG_NAME, "article")),
            "img (전체)": len(driver.find_elements(By.TAG_NAME, "img")),
            "article.card-search-item": len(driver.find_elements(By.CSS_SELECTOR, "article.card-search-item")),
        }
        page_source_len = len(driver.page_source)

        raise TimeoutError(
            f"{timeout}초 안에 검색 결과 카드를 찾지 못했습니다.\n"
            f"  페이지 제목: {title}\n"
            f"  현재 URL: {driver.current_url}\n"
            f"  스크린샷: {screenshot_path}\n"
            f"  태그 개수: {counts}\n"
            f"  page_source 길이: {page_source_len}\n"
            "  → 스크린샷(debug/crawl_timeout.png) 열어서 실제로 뭐가 떠 있는지 확인하세요."
        ) from None

    time.sleep(1)
    n = len(driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR))
    print(f"카드 {n}개 확인")

In [ ]:
def to_original_size(url: str) -> str:
    """썸네일 URL을 고해상도로 변환 (w=480 → w=1200)."""
    url = re.sub(r"w=\d+", "w=1200", url)
    url = re.sub(r"h=\d+", "h=1200", url)
    return url


def run_with_retry(func, *args, retries=3, delay=5, **kwargs):
    """오래 크롤링하면 크롬이 느려져 드라이버 명령이 ReadTimeoutError로 끊길 때가
    있어, 완전히 죽는 대신 잠깐 쉬었다가 같은 명령을 재시도한다."""
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  [드라이버 응답 지연, 재시도 {attempt}/{retries}] {e}")
            time.sleep(delay)


def get_pending_urls(driver, downloaded):
    """위→아래 순서로, 아직 다운로드하지 않은 img.image src URL."""
    pending = []
    for img in driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR):
        src = img.get_attribute("src")
        if not src or not src.startswith("http"):
            continue
        url = to_original_size(src)
        if url not in downloaded and url not in pending:
            pending.append(url)
    return pending


def scroll_once(driver, pause=SCROLL_PAUSE, steps=6, step_pixels=600):
    """실제 마우스 휠처럼 여러 번 조금씩 스크롤(ActionChains). window.scrollTo/
    scrollIntoView의 JS 강제 이동으로는 이 페이지의 무한스크롤이 안 트리거돼서
    사람이 휠 굴리는 것과 더 비슷한 방식으로 바꿈."""
    before = len(driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR))

    actions = ActionChains(driver)
    for _ in range(steps):
        actions.scroll_by_amount(0, step_pixels)
        actions.pause(0.3)
    actions.perform()

    time.sleep(pause)
    after = len(driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR))
    return after > before


def build_existing_hashes(images_root=Path("images")) -> set:
    """images/ 아래 모든 폴더(black, gray, white, final, pinterest_* 등)를 훑어서
    이미 갖고 있는 사진의 MD5 콘텐츠 해시 집합을 만든다.
    → 다른 폴더/다른 URL로 이미 크롤링된 사진을 이번 크롤링에서 또 저장하는 것을 방지."""
    hashes = set()
    if not images_root.exists():
        return hashes
    for path in images_root.rglob("*"):
        if path.is_file():
            try:
                hashes.add(hashlib.md5(path.read_bytes()).hexdigest())
            except OSError:
                pass
    return hashes


def download_one(session, url, save_dir, existing_hashes=None):
    """URL에서 이미지 1장 다운로드. 파일명은 URL 해시지만, 중복 판단은
    바이트 내용의 MD5로 해서 다른 폴더에 이미 있는 사진도 걸러낸다."""
    r = session.get(url, timeout=15)
    r.raise_for_status()

    content_hash = hashlib.md5(r.content).hexdigest()
    if existing_hashes is not None and content_hash in existing_hashes:
        return None, "duplicate"

    ext = Path(urlparse(url).path).suffix or ".jpg"
    name = hashlib.md5(url.encode()).hexdigest()[:12] + ext
    path = save_dir / name

    if path.exists():
        return name, "duplicate"

    path.write_bytes(r.content)
    if existing_hashes is not None:
        existing_hashes.add(content_hash)
    return name, "new"


def crawl_row_by_row(driver, save_dir, max_count=MAX_IMAGES, row_size=ROW_SIZE):
    """위에서부터 한 줄(4개)씩 다운로드, 없을 때만 스크롤. images/ 전체에 이미 있는
    사진(다른 폴더 포함)은 콘텐츠 해시로 건너뛴다."""
    if "Access Denied" in driver.title:
        raise RuntimeError("오늘의집 접근 차단됨. headless 대신 최소화 창 모드로 실행 중인지 확인하세요.")

    session = requests.Session()  # 연결 재사용으로 다운로드 속도 향상
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://ohou.se/",
    })

    print("기존 images/ 전체에서 중복 체크용 해시 만드는 중...")
    existing_hashes = build_existing_hashes()
    print(f"기존 이미지 {len(existing_hashes)}장 기준으로 중복 제외하며 진행합니다.")

    downloaded = set()  # 이미 처리한 URL (이번 실행 내에서만)
    saved = 0
    skipped = 0
    row_num = 0
    no_new_scrolls = 0  # 연속 스크롤 실패 횟수

    while saved < max_count:
        pending = run_with_retry(get_pending_urls, driver, downloaded)

        if not pending:
            print(f"다운로드할 이미지 없음 → 스크롤 (재시도 {no_new_scrolls + 1}/{MAX_NO_NEW_SCROLLS})")
            if not run_with_retry(scroll_once, driver):
                no_new_scrolls += 1
                if no_new_scrolls >= MAX_NO_NEW_SCROLLS:
                    print("더 이상 새 콘텐츠가 없습니다.")
                    break
            else:
                no_new_scrolls = 0
            continue

        no_new_scrolls = 0
        row_num += 1
        batch = pending[:row_size]  # 한 줄 = ROW_SIZE장
        print(f"\n[{row_num}줄] {len(batch)}장 처리")

        for url in batch:
            if saved >= max_count:
                break
            try:
                name, status = download_one(session, url, save_dir, existing_hashes)
                downloaded.add(url)
                if status == "new":
                    saved += 1
                    print(f"  저장: {name} ({saved}/{max_count})")
                else:
                    skipped += 1
                    print(f"  건너뜀(중복): {name}")
            except Exception as e:
                downloaded.add(url)
                print(f"  실패: {url[:60]}... ({e})")

    print(f"\n신규 {saved}장 저장, {skipped}장 건너뜀(중복) → {save_dir.resolve()}")

In [ ]:
# === 실행 (검색어 8개를 순서대로 순회) ===
driver = create_driver()  # Chrome 최소화 모드 (headless는 Access Denied), 전체 검색어 공용

try:
    for i, query in enumerate(QUERIES, 1):
        search_url = build_search_url(query)
        save_dir = build_save_dir(query)
        save_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*50}\n[{i}/{len(QUERIES)}] 검색어: {query} → {save_dir}\n{'='*50}")
        try:
            driver.get(search_url)
            wait_for_cards(driver)
            print(f"페이지 로드 완료: {driver.title}")
            crawl_row_by_row(driver, save_dir=save_dir)
        except Exception as e:
            # 검색어 하나가 실패해도(차단/타임아웃 등) 나머지 검색어는 이어서 진행
            print(f"[{query}] 크롤링 실패, 다음 검색어로 넘어갑니다: {e}")
finally:
    driver.quit()  # 브라우저 종료